In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 18:00:12 WARN Utils: Your hostname, Mateos-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.14 instead (on interface en0)
26/03/11 18:00:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 18:00:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'4.1.1'

In [11]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-11 18:15:55--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.198, 3.170.186.229, 3.170.186.111, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.198|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  49.1MB/s    in 1.4s    

2026-03-11 18:15:57 (49.1 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [6]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [14]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

df = df.repartition(4)

df.write.mode("overwrite").parquet('data/pq/yellow/2025/11/')

In [16]:
df = spark.read.parquet('data/pq/fhvhv/2021/02/')


[Stage 0:>                                                          (0 + 1) / 1]



**Q3**: How many taxi trips were there on February 15?

In [16]:
spark.sql("""
    SELECT COUNT(*) 
    FROM parquet.`data/pq/yellow/2025/11/`
    WHERE DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [24]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .filter("pickup_date = '2021-02-15'") \
    .count()

367170

In [25]:
df.registerTempTable('fhvhv_2021_02')

In [28]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    fhvhv_2021_02
WHERE
    to_date(pickup_datetime) = '2021-02-15';
""").show()

+--------+
|count(1)|
+--------+
|  367170|
+--------+




[Stage 20:==============>                                           (1 + 3) / 4]



**Q4**: Longest trip for each day

In [24]:
spark.sql("""
SELECT PULocationID,
    COUNT(*) NUM
FROM 
    parquet.`data/pq/yellow/2025/11/`
GROUP BY PULocationID
ORDER BY NUM ASC;
""").show()

+------------+---+
|PULocationID|NUM|
+------------+---+
|           5|  1|
|         105|  1|
|          84|  1|
|         187|  3|
|         111|  4|
|         109|  4|
|         204|  4|
|         199|  4|
|           2|  5|
|         251| 12|
|          59| 14|
|         176| 14|
|         245| 14|
|         172| 14|
|         253| 15|
|          27| 16|
|         206| 17|
|          30| 18|
|         156| 21|
|         118| 22|
+------------+---+
only showing top 20 rows


In [36]:
df \
    .withColumn('duration', df.dropoff_datetime.cast('long') - df.pickup_datetime.cast('long')) \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .groupBy('pickup_date') \
        .max('duration') \
    .orderBy('max(duration)', ascending=False) \
    .limit(5) \
    .show()

+-----------+-------------+
|pickup_date|max(duration)|
+-----------+-------------+
| 2021-02-11|        75540|
| 2021-02-17|        57221|
| 2021-02-20|        44039|
| 2021-02-03|        40653|
| 2021-02-19|        37577|
+-----------+-------------+




[Stage 38:==================================================>   (187 + 4) / 200]



In [22]:
spark.sql("""
SELECT
    to_date(tpep_dropoff_datetime) AS pickup_date,
    MAX(TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime))/60 AS duration
FROM 
    parquet.`data/pq/yellow/2025/11/`
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 10;
""").show()

+-----------+-----------------+
|pickup_date|         duration|
+-----------+-----------------+
| 2025-11-30|90.63333333333334|
| 2025-11-06|             76.2|
| 2025-11-10|69.28333333333333|
| 2025-11-21|67.06666666666666|
| 2025-11-25|63.36666666666667|
| 2025-11-03|56.36666666666667|
| 2025-11-29|48.13333333333333|
| 2025-11-24|45.43333333333333|
| 2025-11-28|44.03333333333333|
| 2025-11-07|42.71666666666667|
+-----------+-----------------+



**Q5**: Most frequent `dispatching_base_num`

How many stages this spark job has?



In [44]:
spark.sql("""
SELECT
    dispatching_base_num,
    COUNT(1)
FROM 
    fhvhv_2021_02
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()


[Stage 73:>                                                         (0 + 4) / 4]



+--------------------+--------+
|dispatching_base_num|count(1)|
+--------------------+--------+
|              B02510| 3233664|
|              B02764|  965568|
|              B02872|  882689|
|              B02875|  685390|
|              B02765|  559768|
+--------------------+--------+




[Stage 74:===================================================>  (189 + 5) / 200]



In [46]:
df \
    .groupBy('dispatching_base_num') \
        .count() \
    .orderBy('count', ascending=False) \
    .limit(5) \
    .show()


[Stage 86:>                                                         (0 + 4) / 4]



+--------------------+-------+
|dispatching_base_num|  count|
+--------------------+-------+
|              B02510|3233664|
|              B02764| 965568|
|              B02872| 882689|
|              B02875| 685390|
|              B02765| 559768|
+--------------------+-------+




[Stage 87:===========================================>          (161 + 5) / 200]



**Q6**: Most common locations pair

In [47]:
df_zones = spark.read.parquet('zones')

In [49]:
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [51]:
df.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

In [50]:
df_zones.registerTempTable('zones')

In [57]:
spark.sql("""
SELECT
    CONCAT(pul.Zone, ' / ', dol.Zone) AS pu_do_pair,
    COUNT(1)
FROM 
    fhvhv_2021_02 fhv LEFT JOIN zones pul ON fhv.PULocationID = pul.LocationID
                      LEFT JOIN zones dol ON fhv.DOLocationID = dol.LocationID
GROUP BY 
    1
ORDER BY
    2 DESC
LIMIT 5;
""").show()

+--------------------+--------+
|          pu_do_pair|count(1)|
+--------------------+--------+
|East New York / E...|   45041|
|Borough Park / Bo...|   37329|
| Canarsie / Canarsie|   28026|
|Crown Heights Nor...|   25976|
|Bay Ridge / Bay R...|   17934|
+--------------------+--------+

